# Quantitative X-Ray Diffraction (XRD) Analysis

This notebook demonstrates quantitative analysis of powder X-ray diffraction patterns:
- Ingestion of raw XRD data (CSV and comment-delimited text)
- Bragg's law $d$-spacing calculation: $\lambda = 2d\sin\theta$
- Scherrer crystallite domain size calculation with instrumental broadening correction:
  $$\beta_{\mathrm{corr}} = \sqrt{\beta_{\mathrm{obs}}^2 - \beta_{\mathrm{inst}}^2}, \quad D = \frac{K \lambda}{\beta_{\mathrm{corr}} \cos\theta}$$
- Multi-peak pseudo-Voigt deconvolution using `lmfit` with residual analysis
- Stacked comparative diffractograms with calibrated vertical offsets.

In [1]:
import sys
from pathlib import Path
# Ensure project root is in sys.path
root_path = Path.cwd().parent.parent if Path.cwd().name == 'characterization' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import matplotlib.pyplot as plt
import numpy as np
from parsers.characterization import parse_xrd_file
from modules.characterization.xrd import (
    calculate_d_spacing,
    calculate_crystallite_size_scherrer,
    analyze_xrd_peaks,
    plot_xrd_pattern,
    plot_xrd_stacked,
    plot_xrd_fit,
)
from modules.characterization.common_peaks import fit_peaks
from scripts.utils import apply_publication_style

apply_publication_style()
print('XRD analysis modules loaded successfully.')

XRD analysis modules loaded successfully.


## 1. Single Diffractogram & Peak Analysis
Load pristine Pt/C catalyst diffractogram collected with Cu K$\alpha$ radiation ($\lambda = 1.54060\text{ \AA}$).

In [2]:
data_path = root_path / 'data' / 'examples' / 'characterization' / 'xrd' / 'catalyst_pristine_xrd.csv'
xrd_data = parse_xrd_file(data_path)
print(f'Loaded: {xrd_data.sample_id} ({xrd_data.n_points} points, 2theta: {xrd_data.x.min():.1f} to {xrd_data.x.max():.1f} deg)')

peaks = analyze_xrd_peaks(xrd_data, prominence=50.0)
for p in peaks:
    print(f'Peak {p["peak_index"]}: 2theta = {p["two_theta_deg"]:.2f} deg, d = {p["d_spacing_angstrom"]:.3f} A, Size = {p["crystallite_size_nm"]:.1f} nm')

fig, ax = plot_xrd_pattern(xrd_data, annotate_peaks=True, show_d_spacing=True, title='Pt/C Catalyst Powder XRD')
plt.show()

Loaded: catalyst_pristine_xrd (1401 points, 2theta: 15.0 to 85.0 deg)
Peak 1: 2theta = 16.55 deg, d = 5.352 A, Size = 0.1 nm
Peak 2: 2theta = 18.65 deg, d = 4.754 A, Size = 0.4 nm
Peak 3: 2theta = 20.30 deg, d = 4.371 A, Size = 0.5 nm
Peak 4: 2theta = 21.25 deg, d = 4.178 A, Size = 0.6 nm
Peak 5: 2theta = 22.05 deg, d = 4.028 A, Size = 0.6 nm
Peak 6: 2theta = 22.80 deg, d = 3.897 A, Size = 0.6 nm
Peak 7: 2theta = 23.95 deg, d = 3.713 A, Size = 0.7 nm
Peak 8: 2theta = 25.45 deg, d = 3.497 A, Size = 0.7 nm
Peak 9: 2theta = 27.60 deg, d = 3.229 A, Size = 0.6 nm
Peak 10: 2theta = 31.15 deg, d = 2.869 A, Size = 0.2 nm
Peak 11: 2theta = 33.70 deg, d = 2.657 A, Size = 0.1 nm
Peak 12: 2theta = 34.65 deg, d = 2.587 A, Size = 0.1 nm
Peak 13: 2theta = 36.00 deg, d = 2.493 A, Size = 0.1 nm
Peak 14: 2theta = 39.70 deg, d = 2.269 A, Size = 4.1 nm
Peak 15: 2theta = 46.20 deg, d = 1.963 A, Size = 3.8 nm
Peak 16: 2theta = 50.35 deg, d = 1.811 A, Size = 0.1 nm
Peak 17: 2theta = 51.90 deg, d = 1.760 A, S

C:\Users\mjram\python_graphing\modules\characterization\xrd.py:286: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_20236\1850166229.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Stacked Diffractograms (Pristine vs Treated Catalyst)
Compare crystal phase transformations upon thermal treatment.

In [3]:
treated_path = root_path / 'data' / 'examples' / 'characterization' / 'xrd' / 'catalyst_treated_xrd.txt'
treated_data = parse_xrd_file(treated_path)

fig, ax = plot_xrd_stacked(
    [xrd_data, treated_data],
    labels=['Pristine Pt/C', 'Thermally Treated Pt/C'],
    vertical_offset_fraction=0.35,
    title='Phase Evolution: Pristine vs Thermally Activated'
)
plt.show()

C:\Users\mjram\python_graphing\modules\characterization\xrd.py:338: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_20236\1694588460.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. High-Resolution Peak Deconvolution & Scherrer Size
Deconvolute the Pt(111) reflection around $2\theta = 39.8^\circ$ using a pseudo-Voigt profile.

In [4]:
mask = (xrd_data.x >= 36.0) & (xrd_data.x <= 44.0)
sub_x, sub_y = xrd_data.x[mask], xrd_data.y[mask]
fit_res = fit_peaks(sub_x, sub_y, peak_centers=[39.76], model_type='pseudo_voigt', background='linear')

ctr = fit_res['peaks'][0]['center']
fwhm = fit_res['peaks'][0]['fwhm']
scherrer = calculate_crystallite_size_scherrer(ctr, fwhm, instrumental_fwhm_deg=0.08)
print(f'Pt(111) Reflection: Center = {ctr:.2f} deg, FWHM = {fwhm:.3f} deg')
print(f'Scherrer Domain Size: {scherrer["crystallite_size_nm"]:.2f} nm ({scherrer["crystallite_size_angstrom"]:.1f} A)')

sub_data = xrd_data.copy()
sub_data.x = sub_x
sub_data.y = sub_y
fig, axes = plot_xrd_fit(sub_data, fit_res, title=f'Pt(111) Reflection Deconvolution (Domain Size = {scherrer["crystallite_size_nm"]:.1f} nm)')
plt.show()

Pt(111) Reflection: Center = 39.76 deg, FWHM = 1.994 deg
Scherrer Domain Size: 4.24 nm (42.4 A)


C:\Users\mjram\python_graphing\modules\characterization\xrd.py:407: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\mjram\python_graphing\modules\characterization\xrd.py:407: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_20236\3611616606.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
